# LILaC reproduction (MMCoQA) — Colab driver
Runs `scripts/lilac/pipeline.sh` stage by stage. Each GPU stage is resumable; run `sync.sh push` after every stage.

### 0. Session setup
Runtime → *Change runtime type* → **L4 GPU** (or A100). Store `WANDB_API_KEY` and `HF_TOKEN` in Colab *Secrets* (key icon on the left).

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
for k in ('WANDB_API_KEY', 'HF_TOKEN'):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print('secret not set:', k)
os.environ['WANDB_ENTITY'] = 'YOUR_WANDB_ENTITY'   # <- edit
os.environ['ENV_ROOT'] = '/content/envs'

In [ ]:
%cd /content
!git clone https://github.com/YOUR_GITHUB_USER/mmgraphrag-repro.git 2>/dev/null || (cd mmgraphrag-repro && git pull)
%cd /content/mmgraphrag-repro
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1. Environments (once per VM, ~5–10 min)

In [ ]:
!bash scripts/setup_envs.sh lilac-base mmembed vllm

### 2. Official code + data (restores earlier artefacts from Drive if present)

In [ ]:
!bash scripts/colab/sync.sh pull
!bash scripts/lilac/pipeline.sh fetch data

### 3. Graph construction: Qwen2.5-VL passes (vLLM) + official steps 1–7

In [ ]:
!bash scripts/lilac/pipeline.sh vlm graph && bash scripts/colab/sync.sh push

### 4. MM-Embed corpus embeddings (needs ≥ 20 GB GPU memory)

In [ ]:
!bash scripts/lilac/pipeline.sh embed && bash scripts/colab/sync.sh push

### 5. Query decomposition with several LLMs + sub-query embeddings
On an L4, drop the 32B model: `DECOMPOSERS='qwen2.5-7b=Qwen/Qwen2.5-7B-Instruct qwen2.5-14b-awq=Qwen/Qwen2.5-14B-Instruct-AWQ'`.

In [ ]:
!bash scripts/lilac/pipeline.sh decompose && bash scripts/colab/sync.sh push

### 6. (optional) extra ablation: image nodes without VLM text

In [ ]:
!bash scripts/lilac/pipeline.sh novlm && bash scripts/colab/sync.sh push

### 7. Retrieval experiments → W&B (CPU is enough; can also run locally)

In [ ]:
!bash scripts/lilac/pipeline.sh experiments && bash scripts/colab/sync.sh push

### 8. W&B report pages + LaTeX tables for the written report

In [ ]:
!/content/envs/lilac-base/bin/python scripts/make_wandb_reports.py --share
!/content/envs/lilac-base/bin/python scripts/export_results.py --results results --out results/latex